# Multi-Agent Social Media Assistant — RuntimeAssistantAgent

This notebook demonstrates our newly implemented **Layer 2** `RuntimeAssistantAgent` in action.

We will set up a **Social Media Marketing Assistant** that uses custom tools (e.g., extracting hashtags and checking brand policy violations) under the hood using an autonomous tool-calling loop, all orchestrated natively within the Actor model runtime.

### Instructions to Run
To run this notebook, execute each cell sequentially from top to bottom. No external LLM key is needed as we mock the model client.

## 1. Architectural Overview

```mermaid
flowchart TD
    User([User Proxy]) -->|Send Post Draft| Agent[SocialMediaAssistant]
    Agent -->|1. Run Guardrails| GR[Guardrail Checker]
    Agent -->|2. Think| LLM[LLM Client]
    LLM -->|3. Need Tagging| Tool1[generate_social_tags Tool]
    LLM -->|4. Need Policy Check| Tool2[check_brand_policy Tool]
    Tool1 & Tool2 -->|Return Results| LLM
    LLM -->|5. Think Again| Agent
    Agent -->|6. Return Final Post| User
```

## 2. Setup and Imports

In [1]:
import asyncio
from unittest.mock import AsyncMock, MagicMock

from ravi.core.runtime import LocalRuntime, AgentId, TopicId, MessageContext
from ravi.core.agents.runtime_assistant_agent import RuntimeAssistantAgent
from ravi.core.messages.client_messages import AssistantMessage, ToolCallMessage, UserMessage
from ravi.core.context.base_context import ModelContext
from ravi.core.llm.base_client import BaseModelClient
from ravi.core.tools.base_tool import BaseTool, ToolResult
from ravi.core.guardrails.base_guardrail import BaseGuardrail, GuardrailResult

print("✅ All core imports succeeded!")

✅ All core imports succeeded!


## 3. Define Tools & Guardrails

In [2]:
# Define a custom Tag Extraction tool
class SocialTagTool(BaseTool):
    def __init__(self):
        super().__init__(
            name="generate_social_tags",
            description="Extracts key topics and returns standard marketing hashtags.",
            input_schema={
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "The text to extract tags from"}
                },
                "required": ["text"]
            }
        )
    
    async def execute(self, text: str) -> ToolResult:
        words = text.split()
        tags = [f"#{w.strip('?!.,').lower()}" for w in words if len(w) > 5]
        result_str = " ".join(tags[:3]) if tags else "#marketing #general"
        return ToolResult(output_text=result_str)


# Define a Brand Safety / Moderation tool
class BrandPolicyTool(BaseTool):
    def __init__(self):
        super().__init__(
            name="check_brand_policy",
            description="Checks if post text complies with safety and community standards.",
            input_schema={
                "type": "object",
                "properties": {
                    "text": {"type": "string", "description": "Post content draft"}
                },
                "required": ["text"]
            }
        )
    
    async def execute(self, text: str) -> ToolResult:
        blocked = ["scam", "spam", "crypto-giveaway"]
        violations = [w for w in blocked if w in text.lower()]
        if violations:
            return ToolResult(output_text=f"REJECTED: Contains forbidden keywords: {violations}")
        return ToolResult(output_text="APPROVED: Safe for publishing")

print("✅ Tools defined successfully!")

✅ Tools defined successfully!


## 4. Mock the LLM Client & Context

We mock the model client behavior to simulate a 2-step ReAct thought loop:
1. **Step 1**: The LLM analyzes the draft, realizes it needs to tag the post and check brand safety. It issues two tool calls in parallel (`generate_social_tags` and `check_brand_policy`).
2. **Step 2**: The LLM receives the results (hashtags generated, policy check approved), writes the final social post draft, and returns it to the user.

In [3]:
mock_client = MagicMock(spec=BaseModelClient)
mock_client.generate = AsyncMock()

mock_client.generate.side_effect = [
    # Step 1: LLM decides to call tools
    AssistantMessage(
        content=None,
        tool_calls=[
            ToolCallMessage(
                id="call-tag",
                name="generate_social_tags",
                arguments={"text": "This new modular actor framework is absolutely awesome!"}
            ),
            ToolCallMessage(
                id="call-policy",
                name="check_brand_policy",
                arguments={"text": "This new modular actor framework is absolutely awesome!"}
            )
        ]
    ),
    
    # Step 2: Final response using tool results
    AssistantMessage(
        content=["Draft Approved! Final post: 'This new modular actor framework is absolutely awesome! #modular #framework #awesome'"],
        tool_calls=None
    )
]

# Simple pass-through context
mock_context = MagicMock(spec=ModelContext)
mock_context.build = AsyncMock(return_value=[])

print("✅ Mock LLM & Context configured!")

✅ Mock LLM & Context configured!


## 5. Execute the Loop in the Runtime

In [4]:
async def main():
    # 1. Create and start the runtime
    runtime = LocalRuntime()
    await runtime.start()
    
    # 2. Instantiate tools
    tools = [SocialTagTool(), BrandPolicyTool()]
    
    # 3. Create the Assistant Agent (Layer 2)
    assistant = RuntimeAssistantAgent(
        name="social_media_assistant",
        runtime=runtime,
        model_client=mock_client,
        model_context=mock_context,
        tools=tools,
        system_instructions="You are a helpful social media manager. Always optimize tags and check brand policy."
    )
    
    # 4. Start the agent (auto-registers with Layer 1 runtime!)
    await assistant.start()
    
    # 5. Send message from User Proxy to the Assistant Agent via the runtime
    print("\n--- Sending draft to SocialMediaAssistant ---")
    response = await runtime.send_message(
        "This new modular actor framework is absolutely awesome!",
        recipient=assistant.id
    )
    
    # 6. Print final result
    print("\n--- Final Assistant Response ---")
    print(response)
    
    await runtime.stop()

await main()


--- Sending draft to SocialMediaAssistant ---

--- Final Assistant Response ---
Draft Approved! Final post: 'This new modular actor framework is absolutely awesome! #modular #framework #awesome'


## 6. Conclusion

This notebook demonstrates how cleanly our **Layer 2** `RuntimeAssistantAgent` coordinates:
- Auto-registration with Layer 1 runtime.
- Input guardrails and contextual memory formatting.
- Standardized parallel tool calling executions.
- Output guardrails checks.

This provides developers with a structured, resilient, and enterprise-grade assistant loop while completely retaining the underlying actor model transport mechanics!